In [1]:
import json
import uuid
from sqlalchemy import create_engine

from utils import reset_db, get_session, model_to_dict
from data.models import udahub

/Users/burakkilic/msc-ai/curriculum/nd901-agentic-ai-langchain-langgraph/projects/autonomous-knowledge-agent/.venv/lib/python3.13/site-packages/langgraph/cache/base/__init__.py:8: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


# Udahub Application

## Core Database

**Init DB**

In [2]:
udahub_db = "data/core/udahub.db"

In [3]:
reset_db(udahub_db)

✅ Removed existing data/core/udahub.db
2026-10-14 14:05:35,572 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-10-14 14:05:35,573 INFO sqlalchemy.engine.Engine COMMIT
✅ Recreated data/core/udahub.db with fresh schema


In [4]:
engine = create_engine(f"sqlite:///{udahub_db}", echo=False)
udahub.Base.metadata.create_all(bind=engine)

**Account**

In [5]:
account_id = "cultpass"
account_name = "CultPass Card"

In [6]:
with get_session(engine) as session:
    account = udahub.Account(
        account_id=account_id,
        account_name=account_name,
    )
    session.add(account)

## Integrations

**Knowledge Base**

In [7]:
additional_articles = [
    {
        "title": "Support for a Blocked CultPass Account",
        "content": (
            "Check the customer's verified account status. If the account is "
            "blocked, refer the case for human review. Do not promise to remove "
            "the block, create reservations, or restore access. The stored "
            "blocked flag does not explain why the restriction exists. Preserve "
            "the customer's request and verified findings in the handoff "
            "without inventing a reason or a response deadline."
        ),
        "tags": "account management, blocked account, access, escalation",
    },
    {
        "title": "Password Reset Email Has Not Arrived",
        "content": (
            "Ask the customer to use 'Forgot Password' on the CultPass login "
            "screen and check that they entered their registered email. Suggest "
            "checking the spam folder. If the reset message has not arrived, "
            "retry after 10 minutes, as described in the login guidance. "
            "Persistent reset-email or login problems require human support. "
            "Do not claim that support sent an email or confirmed its delivery."
        ),
        "tags": "technical issues, password reset, email, login",
    },
    {
        "title": "Checking the Current Subscription Status",
        "content": (
            "For a question about the customer's own subscription, retrieve "
            "the verified subscription record for that customer. Report the "
            "stored status and tier as recorded. Subscription status alone "
            "does not establish payment history, the next charge, or unused "
            "experiences. If the lookup fails, explain that verification could "
            "not be completed; do not interpret a technical failure as an "
            "absent subscription. Refer unresolved cases for human review."
        ),
        "tags": "subscription, account management, status, plan",
    },
    {
        "title": "Monthly Allowance and Remaining Experiences",
        "content": (
            "Use the verified customer-specific subscription record when "
            "discussing its stored monthly allowance. A monthly allowance "
            "is not a remaining-use balance. This case does not define a "
            "billing-period counting rule that can convert reservations into "
            "remaining experiences. Do not subtract reservation counts from "
            "the allowance. If the customer's stored quota conflicts with "
            "the general subscription guidance, preserve the discrepancy "
            "and request human review instead of choosing an unsupported value."
        ),
        "tags": "subscription, monthly quota, allowance, usage",
    },
    {
        "title": "Finding and Checking an Existing Reservation",
        "content": (
            "Retrieve reservations only for the customer linked to the support "
            "case. Use a reservation ID when available, or ask for event title, "
            "date, or location to identify the intended booking. If several "
            "records match a request for one booking, ask which booking the "
            "customer means. Report the verified reservation status and "
            "associated event details. Do not claim that a reservation was "
            "created, changed, or cancelled by the support assistant."
        ),
        "tags": "reservation, booking status, account lookup, clarification",
    },
    {
        "title": "Checking an Experience's Date and Location",
        "content": (
            "Use the stored experience details to answer questions about an "
            "event's date, time, location, description, or premium designation. "
            "An experience ID identifies one record; a title, date, or location "
            "may match several experiences. Ask for clarification when the "
            "intended event is unclear. Report the details actually stored, "
            "without inventing a timezone, transport arrangement, accessibility "
            "feature, or venue rule that the available information does not establish."
        ),
        "tags": "experience information, event date, venue, location",
    },
    {
        "title": "Checking Available Places Before Booking",
        "content": (
            "Retrieve the intended experience and report its stored available-place "
            "count. Available places are a snapshot and do not prove that the "
            "customer has a reservation. To make a booking, use the CultPass app, "
            "browse the experience catalogue, and tap 'Reserve', following the "
            "reservation guidance. The support assistant cannot hold places, "
            "increase capacity, or confirm that a booking action occurred. "
            "Do not promise availability beyond the verified record."
        ),
        "tags": "experience information, availability, capacity, reservation",
    },
    {
        "title": "Premium Experiences and Additional Charges",
        "content": (
            "Some premium experiences may require an additional fee, which the "
            "supplied subscription guidance says is visible in the app during "
            "reservation. Check the intended experience's premium designation "
            "when the question concerns a particular event. That designation "
            "does not establish an exact price or prove that a charge occurred. "
            "Direct the customer to the displayed app information for the fee; "
            "refer unresolved charge questions to human support without "
            "inventing an amount."
        ),
        "tags": "billing, premium experiences, fees, pricing",
    },
    {
        "title": "Refund Requests and Unexplained Charges",
        "content": (
            "Refund offers require support-lead approval, as stated in the "
            "cancellation guidance. Preserve the customer's refund request "
            "or charge concern for human review. The available CultPass records "
            "do not include a payment ledger, so subscription or reservation "
            "data cannot verify a debit amount, payment failure, or completed "
            "refund. Do not promise eligibility, a refund amount, processing "
            "time, or that money has been returned."
        ),
        "tags": "billing, refund, payment query, escalation",
    },
    {
        "title": "Cancellation and Pause Requests: What Support Can Confirm",
        "content": (
            "Customers can manage cancellation or pause through the CultPass app "
            "under 'My Account' and 'Manage Plan'. The supplied guidance states "
            "that cancellation takes effect at the end of the billing cycle "
            "and that pausing preserves user data. The support assistant can "
            "explain this guidance and inspect the stored subscription, but "
            "cannot execute either change. Requests needing action or verification "
            "beyond these records require human review. Do not claim an effective "
            "date, completed cancellation, or automatic change without "
            "supporting evidence."
        ),
        "tags": "subscription, cancellation, pause, account management",
    },
]

# Validate the authored records before saving.
if any(
    not article[field].strip()
    for article in additional_articles
    for field in ("title", "content", "tags")
):
    raise ValueError("Every article needs a title, content and tags.")

titles = [article["title"] for article in additional_articles]
if len(set(titles)) != len(titles):
    raise ValueError("Article titles must be distinct.")

article_path = "data/external/cultpass_articles.jsonl"

with open(article_path, "r", encoding="utf-8") as file:
    original_text = file.read()

existing_articles = [
    json.loads(line)
    for line in original_text.splitlines()
    if line.strip()
]

# Prevent duplicates when this cell is run again.
existing_titles = {article["title"] for article in existing_articles}
new_articles = [
    article
    for article in additional_articles
    if article["title"] not in existing_titles
]

# api: mode "a" appends without overwriting the supplied articles.
with open(article_path, "a", encoding="utf-8") as file:
    if new_articles and original_text and not original_text.endswith("\n"):
        file.write("\n")

    for article in new_articles:
        # api: dumps serializes each dictionary as one JSONL record.
        file.write(json.dumps(article, ensure_ascii=False) + "\n")

print(f"Added {len(new_articles)} articles.")
print(f"Total articles: {len(existing_articles) + len(new_articles)}")

Added 0 articles.
Total articles: 14


In [8]:
cultpass_articles = []

with open('data/external/cultpass_articles.jsonl', 'r', encoding='utf-8') as f:
    for line in f:
        cultpass_articles.append(json.loads(line))

In [9]:
cultpass_articles

[{'title': 'How to Reserve a Spot for an Event',
  'content': 'If a user asks how to reserve an event:\n\n- Guide them to the CultPass app\n- Instruct them to browse the experience catalog and tap \'Reserve\'\n- If it\'s a premium or limited event, check if reservation confirmation is required via email\n- Remind them to arrive at least 15 minutes early with their QR code visible\n\n**Suggested phrasing:**\n"You can reserve an experience by opening the CultPass app, selecting your desired event, and tapping \'Reserve\'. Be sure to arrive 15 minutes early with your QR code ready."',
  'tags': 'reservation, events, booking, attendance'},
 {'title': "What's Included in a CultPass Subscription",
  'content': 'Each user is entitled to 4 cultural experiences per month, which may include:\n- Art exhibitions\n- Museum entries\n- Music concerts\n- Film screenings and more\n\nSome premium experiences may require an additional fee (visible in the app).\n\n**Suggested phrasing:**\n"Your CultPass s

In [10]:
if len(cultpass_articles) < 14:
    raise AssertionError("You should load the articles with at least 14 records")

In [11]:
with get_session(engine) as session:
    kb = []
    for article in cultpass_articles:
        knowledge = udahub.Knowledge(
            article_id=str(uuid.uuid4()),
            account_id=account_id,
            title=article["title"],
            content=article["content"],
            tags=article["tags"]
        )
        kb.append(knowledge)
    session.add_all(kb) 
    

**Ticket**

In [12]:
cultpass_users = []

with open('data/external/cultpass_users.jsonl', 'r', encoding='utf-8') as f:
    for line in f:
        cultpass_users.append(json.loads(line))

In [13]:
ticket_info = {
    "status": "open",
    "content": "I can't log in to my Cultpass account.",
    "owner_id": cultpass_users[0]["id"],
    "owner_name": cultpass_users[0]["name"],
    "role": "user",
    "channel": "chat",
    "tags": "login, access",
}

In [14]:
with get_session(engine) as session:
    user = session.query(udahub.User).filter_by(
        account_id=account_id,
        external_user_id=ticket_info["owner_id"],
    ).first()

    if not user:
        user = udahub.User(
            user_id=str(uuid.uuid4()),
            account_id=account_id,
            external_user_id=ticket_info["owner_id"],
            user_name=ticket_info["owner_name"],
        )
    
    ticket = udahub.Ticket(
        ticket_id=str(uuid.uuid4()),
        account_id=account_id,
        user_id=user.user_id,
        channel=ticket_info["channel"],
    )
    metadata = udahub.TicketMetadata(
        ticket_id=ticket.ticket_id,
        status=ticket_info["status"],
        main_issue_type=None,
        tags=ticket_info["tags"],
    )

    first_message = udahub.TicketMessage(
        message_id=str(uuid.uuid4()),
        ticket_id=ticket.ticket_id,
        role=ticket_info["role"],
        content=ticket_info["content"],
    )

    session.add_all([user, ticket, metadata, first_message])


In [15]:
# Additional email cases: resolution, returning customer, tools, escalation and follow-up.
from datetime import datetime, timedelta

# Explicit dates preserve scenario order even when UUIDs sort differently.
demo_start = datetime(2026, 1, 1, 12)
unblocked_user = next(
    user for user in cultpass_users if user.get("is_blocked") is False
)

ticket_specs = [
    (
        "resolution-and-preference",
        "How do I reset my password? Please keep replies concise.",
    ),
    (
        "returning-customer",
        "I need the password-reset steps again.",
    ),
    (
        "subscription-lookup",
        "What is my current subscription status?",
    ),
    (
        "unsupported-request",
        "Can you renew my passport?",
    ),
    (
        "same-ticket-follow-up",
        "I need help with CultPass, but I have not described the problem yet.",
    ),
    (
        "no-relevant-article",
        "How do I update the email address registered on my CultPass account?",
    ),
]

demo_tickets = []

with get_session(engine) as session:
    support_user = session.query(udahub.User).filter_by(
        account_id=account_id,
        external_user_id=unblocked_user["id"],
    ).first()

    if support_user is None:
        support_user = udahub.User(
            user_id=str(uuid.uuid4()),
            account_id=account_id,
            external_user_id=unblocked_user["id"],
            user_name=unblocked_user["name"],
        )
        session.add(support_user)

    for index, (scenario, content) in enumerate(ticket_specs):
        created_at = demo_start + timedelta(minutes=index)
        new_ticket_id = str(uuid.uuid4())

        session.add_all([
            udahub.Ticket(
                ticket_id=new_ticket_id,
                account_id=account_id,
                user_id=support_user.user_id,
                channel="email",
                created_at=created_at,
            ),
            udahub.TicketMetadata(
                ticket_id=new_ticket_id,
                status="open",
                main_issue_type=None,
                tags=scenario,
            ),
            udahub.TicketMessage(
                message_id=str(uuid.uuid4()),
                ticket_id=new_ticket_id,
                role="user",
                content=content,
                created_at=created_at,
            ),
        ])

        # The second stored message is processed only if clarification leaves this ticket open.
        if scenario == "same-ticket-follow-up":
            follow_up = "My problem is signing in. How do I reset my password?"
            session.add(udahub.TicketMessage(
                message_id=str(uuid.uuid4()), ticket_id=new_ticket_id,
                role="user", content=follow_up,
                created_at=created_at + timedelta(seconds=30),
            ))

        demo_tickets.append({
            "scenario": scenario,
            "ticket_id": new_ticket_id,
            "customer_id": unblocked_user["id"],
            "message": content,
        })

for item in demo_tickets:
    print(json.dumps(item, ensure_ascii=False))

{"scenario": "resolution-and-preference", "ticket_id": "24df20ba-7cd9-443e-9de3-cac4e2478375", "customer_id": "f556c0", "message": "How do I reset my password? Please keep replies concise."}
{"scenario": "returning-customer", "ticket_id": "0cf56053-614f-4b59-8fe5-90f6a3f8b7a5", "customer_id": "f556c0", "message": "I need the password-reset steps again."}
{"scenario": "subscription-lookup", "ticket_id": "7fd7fc25-ad69-4e92-a503-c0a85fe97b57", "customer_id": "f556c0", "message": "What is my current subscription status?"}
{"scenario": "unsupported-request", "ticket_id": "8049a2ef-bb10-4a7a-b4b7-ed21ac344739", "customer_id": "f556c0", "message": "Can you renew my passport?"}
{"scenario": "same-ticket-follow-up", "ticket_id": "d0e6f5a4-c82e-49fb-84aa-6915ec699993", "customer_id": "f556c0", "message": "I need help with CultPass, but I have not described the problem yet."}
{"scenario": "no-relevant-article", "ticket_id": "cb43e6c4-7976-46bb-9b9a-a30338869d3b", "customer_id": "f556c0", "messag

# Tests

In [16]:
with get_session(engine) as session:
    account = session.query(udahub.Account).filter_by(
        account_id=account_id
    ).first()
    print(account)

<Account(account_id='cultpass', account_name='CultPass Card')>


In [17]:
with get_session(engine) as session:
    account = session.query(udahub.Account).filter_by(
        account_id=account_id
    ).first()
    for article in account.knowledge_articles:
        print(article)

<Knowledge(article_id='d1fd4d73-2fe2-4b0d-b1f1-1505c587c7f1', title='How to Reserve a Spot for an Event')>
<Knowledge(article_id='0d139ec7-2cad-450f-87e2-68fa7e119ae9', title='What's Included in a CultPass Subscription')>
<Knowledge(article_id='79a018b1-8c73-47eb-b3a3-80b73ebfb64a', title='How to Cancel or Pause a Subscription')>
<Knowledge(article_id='08d3bd4e-bf0b-480a-a935-5a54c99f23f1', title='How to Handle Login Issues?')>
<Knowledge(article_id='67e9193e-ad84-4be5-9a79-ffdc8438883d', title='Support for a Blocked CultPass Account')>
<Knowledge(article_id='af239b86-13ed-45dd-a8d4-ed9df1c337fa', title='Password Reset Email Has Not Arrived')>
<Knowledge(article_id='97606942-9143-4a8c-a0b2-21d9478fafd9', title='Checking the Current Subscription Status')>
<Knowledge(article_id='921b8eae-2509-4676-8c08-6bfb4d80f372', title='Monthly Allowance and Remaining Experiences')>
<Knowledge(article_id='6c16bbe2-274a-47be-8896-6d28560342b8', title='Finding and Checking an Existing Reservation')>
<K

In [18]:
with get_session(engine) as session:
    users = session.query(udahub.User).all()
    for user in users:
        print(user)

<User(user_id='19376918-b066-4bb1-bc6c-b1c9f8ec7fef', user_name='Alice Kingsley', external_user_id='a4ab87')>
<User(user_id='0b460254-6b3f-41ea-80c0-b5300aff41cd', user_name='Bob Stone', external_user_id='f556c0')>


In [19]:
with get_session(engine) as session:
    user = session.query(udahub.User).filter_by(
        account_id=account_id,
        external_user_id=ticket_info["owner_id"],
    ).first()
    
    ticket:udahub.Ticket = user.tickets[0]
    for message in ticket.messages:
        print(message)

<TicketMessage(message_id='29c55dea-252d-489a-ac44-53b1294eefa6', role='user', content='I can't log in to my Cultpass ...')>
